LAB: Transformación optima de la variable dependiente en modelos de regresión
===

En el desarrollo de modelos de regresión siempre se debería verificar si la precisión del pronóstico puede mejorar mediante una transformación no lineal de la variable dependiente. El objetivo de este laboratorio es determinar si una transformación de Box-Cox permite mejorar la precisión del modelo para un conjunto de datos. 

## Descripción del conjunto de datos

Una compañía de seguros desea pronósticar los gastos médicos de la población asegurada con el fin de recolectar un valor superior en ingresos, tal que le permita obtener utilidades. Los costos son difíciles de pronósticar ya que las condiciones más costosas son más raras y parecen aleatorias; y que ciertas condiciones son más probables para ciertos segmentos de la población (infarto en personas obesas y cáncer en fumadores).

El objetivo es usar una base de datos con 1338 registros de gastos médicos hipotéticos para pacientes de EU con el fin de estimar los costos para determinados segmentos de la población, la cual se encuentra disponible en

https://raw.githubusercontent.com/jdvelasq/datalabs/master/datasets/insurance.csv

La información registrada es la siguiente:

* Age: entero hasta 64.

* Sex: male, female.

* bmi: Body mass index.

* children: entero indicando la cantidad de hijos/dependientes cubiertos por el plan de salud.

* smoker: yes, no.

* region: northest, southeast, southwest, northwest.

* charges: costos.


In [1]:
import warnings

warnings.filterwarnings("ignore")

In [2]:
#
# Realice la lectura de datos e imprima la cantidad de registros leidos
#
# Rta/
# 1338
#
import pandas as pd

#
# Lectura del archivo y verificación
#
df = pd.read_csv(
    "https://raw.githubusercontent.com/jdvelasq/datalabs/master/datasets/insurance.csv",
    sep=",",         # separador de campos
    thousands=None,  # separador de miles para números
    decimal=".",     # separador de los decimales para números
)  

len(df)

1338

In [3]:
#
# Las columnas sex, smoker, region son strings.
# Se convierten a variables categoricas usando
# LabelEncoder de scikit-learn. Imprima los tipos
# de las columnas como resultado
#
# Rta/
# age                int64
# sex               object
# bmi              float64
# children           int64
# smoker            object
# region            object
# charges          float64
# sex_factor         int64
# smoker_factor      int64
# region_factor      int64
# dtype: object
#

from sklearn import preprocessing

#
# Entrena los codificadores
#
encoder_sex = preprocessing.LabelEncoder().fit(df.sex)
encoder_smoker = preprocessing.LabelEncoder().fit(df.smoker)
encoder_region = preprocessing.LabelEncoder().fit(df.region)

#
# Genera las variables transformadas
#
df["sex_factor"] = encoder_sex.transform(df.sex)
df["smoker_factor"] = encoder_smoker.transform(df.smoker)
df["region_factor"] = encoder_region.transform(df.region)

df.dtypes

age                int64
sex               object
bmi              float64
children           int64
smoker            object
region            object
charges          float64
sex_factor         int64
smoker_factor      int64
region_factor      int64
dtype: object

In [4]:
#
# Para aplicar la transformación de Box-Cox la escala 
# de los datos puede generar problemas. Divida la columna
# charges por 10000 y luego sumele 2.0. 
# Imprima los primeros cinco valores de la columna
#
# Rta/
# 0    3.688492
# 1    2.172555
# 2    2.444946
# 3    4.198447
# 4    2.386686
# Name: charges, dtype: float64
#        
df['charges'] = df.charges.map(lambda w: w/10000. + 2)
df.charges[:5]

0    3.688492
1    2.172555
2    2.444946
3    4.198447
4    2.386686
Name: charges, dtype: float64

In [5]:
#
# Use los primeros 1000 datos para estimación de parámetros
# y los 338 restantes para validación
#
df_train = df[:1000]
df_test = df[1000:]

## Pronóstico usando un modelo de regresión lineal

In [6]:
#
# Estime un modelo lineal que use todas las variables
# explicativas (numéricas) y reporte el MSE para la 
# muestra de entrenamiento
#
# Rta/
# 0.3492548046438354
#
from sklearn import linear_model

# Crea el modelo
regr = linear_model.LinearRegression()

columns = [
    "age",
    "bmi",
    "children",
    "sex_factor",
    "smoker_factor",
    "region_factor",
]

# Obtiene las variables independientes
X_train = df_train[columns]

# Obtiene la variable dependientes
y_train_true = df_train.charges

# Calibra el modelo
regr.fit(X_train, y_train_true)

y_train_pred = regr.predict(X_train)


#
# MSE para la muestra de entrenamiento
#
from sklearn.metrics import mean_squared_error

mean_squared_error(y_train_true, y_train_pred)

0.3492548046438354

In [7]:
#
# Reporte el MSE para la muestra de prueba
#
# Rta/
# 0.4150890835383797
#
X_test = df_test[columns]
y_test_true = df_test.charges
y_test_pred = regr.predict(X_test)
mean_squared_error(y_test_true, y_test_pred)

0.4150890835383797

In [8]:
#
# Use la siguiente implementación de la
# transformación de Box-Cox
#
import numpy as np


def boxcox(z, Lambda):
    if Lambda == 0:
        return np.log(z)
    return (np.power(z, Lambda) - 1.0) / Lambda


def boxcox_inv(z, Lambda):
    if Lambda == 0:
        return np.exp(z)
    return np.power(Lambda * z + 1.0, 1.0 / Lambda)

In [15]:
#
# Evalue lambdas entre 0.00 y 1.00 con incrementos de 0.01.
# Busque el lambda que genera el mejor pronóstico para la
# muestra de prueba.
# Reporte el lambda y los MSE de entrenamiento y prueba
# cada vez que se obtenga un MSE de prueba mejor
#
# Rta/
# 0.00  0.3404  0.4145
# 0.01  0.3401  0.4141
# 0.02  0.3399  0.4137
# 0.03  0.3397  0.4133
# 0.04  0.3395  0.4130
# 0.05  0.3393  0.4126
# 0.06  0.3391  0.4123
# 0.07  0.3389  0.4120
# 0.08  0.3387  0.4117
# 0.09  0.3386  0.4113
# 0.10  0.3384  0.4110
# 0.11  0.3382  0.4107
# 0.12  0.3381  0.4104
# 0.13  0.3379  0.4102
# 0.14  0.3378  0.4099
# 0.15  0.3377  0.4096
# 0.16  0.3376  0.4094
# 0.17  0.3374  0.4091
# 0.18  0.3373  0.4089
# 0.19  0.3372  0.4087
# 0.20  0.3371  0.4084
# 0.21  0.3371  0.4082
# 0.22  0.3370  0.4080
# 0.23  0.3369  0.4078
# 0.24  0.3368  0.4076
# 0.25  0.3368  0.4074
# 0.26  0.3367  0.4072
# 0.27  0.3367  0.4071
# 0.28  0.3366  0.4069
# 0.29  0.3366  0.4068
# 0.30  0.3366  0.4066
# 0.31  0.3365  0.4065
# 0.32  0.3365  0.4063
# 0.33  0.3365  0.4062
# 0.34  0.3365  0.4061
# 0.35  0.3365  0.4060
# 0.36  0.3365  0.4059
# 0.37  0.3365  0.4058
# 0.38  0.3365  0.4057
# 0.39  0.3366  0.4056
# 0.40  0.3366  0.4056
# 0.41  0.3366  0.4055
# 0.42  0.3367  0.4054
# 0.43  0.3367  0.4054
# 0.44  0.3368  0.4053
# 0.45  0.3368  0.4053
# 0.46  0.3369  0.4053
# 0.47  0.3370  0.4053
# 0.48  0.3370  0.4052
# 0.49  0.3371  0.4052
#

mse_test_opt = None
Lambda_opt = None

for Lambda in np.linspace(start=0, stop=1.0, num=101):

    y_train_true_transf = boxcox(y_train_true, Lambda)
    y_test_true_transf = boxcox(y_test_true, Lambda)

    regr.fit(X_train, y_train_true_transf)

    y_train_pred_transf = regr.predict(X_train)
    y_test_pred_transf = regr.predict(X_test)

    y_train_pred = boxcox_inv(y_train_pred_transf, Lambda)
    y_test_pred = boxcox_inv(y_test_pred_transf, Lambda)

    mse_train = mean_squared_error(y_train_true, y_train_pred)
    mse_test = mean_squared_error(y_test_true, y_test_pred)

    if mse_test_opt is None or mse_test < mse_test_opt:
        mse_test_opt = mse_test
        Lambda_opt = Lambda
        print("# {:4.2f}  {:5.4f}  {:5.4f}".format(Lambda, mse_train, mse_test))

# 0.00  0.3404  0.4145
# 0.01  0.3401  0.4141
# 0.02  0.3399  0.4137
# 0.03  0.3397  0.4133
# 0.04  0.3395  0.4130
# 0.05  0.3393  0.4126
# 0.06  0.3391  0.4123
# 0.07  0.3389  0.4120
# 0.08  0.3387  0.4117
# 0.09  0.3386  0.4113
# 0.10  0.3384  0.4110
# 0.11  0.3382  0.4107
# 0.12  0.3381  0.4104
# 0.13  0.3379  0.4102
# 0.14  0.3378  0.4099
# 0.15  0.3377  0.4096
# 0.16  0.3376  0.4094
# 0.17  0.3374  0.4091
# 0.18  0.3373  0.4089
# 0.19  0.3372  0.4087
# 0.20  0.3371  0.4084
# 0.21  0.3371  0.4082
# 0.22  0.3370  0.4080
# 0.23  0.3369  0.4078
# 0.24  0.3368  0.4076
# 0.25  0.3368  0.4074
# 0.26  0.3367  0.4072
# 0.27  0.3367  0.4071
# 0.28  0.3366  0.4069
# 0.29  0.3366  0.4068
# 0.30  0.3366  0.4066
# 0.31  0.3365  0.4065
# 0.32  0.3365  0.4063
# 0.33  0.3365  0.4062
# 0.34  0.3365  0.4061
# 0.35  0.3365  0.4060
# 0.36  0.3365  0.4059
# 0.37  0.3365  0.4058
# 0.38  0.3365  0.4057
# 0.39  0.3366  0.4056
# 0.40  0.3366  0.4056
# 0.41  0.3366  0.4055
# 0.42  0.3367  0.4054
# 0.43  0.3

In [10]:
# 
# Realice un gráfico de scatter que compare los 
# charges reales vs los pronósticos para las muestras
# de entrenamiento y prueba
#

array([3.47684643, 1.32569361, 1.64823829, 1.3600976 , 1.53882189,
       1.38936305, 2.08434115, 1.78574175, 1.83223751, 2.20740326,
       1.31112406, 4.60905917, 1.39135802, 2.51981535, 4.18038061,
       1.00122267, 2.24735556, 1.17919097, 2.46195233, 3.98667955,
       2.59144306, 1.57956393, 1.29509476, 4.18814638, 1.72604021,
       2.32187467, 2.24155095, 2.38851746, 0.95985224, 4.1295472 ,
       3.78464772, 1.15638652, 1.32803404, 2.34138681, 4.01180352,
       0.91331843, 2.62190035, 1.01886628, 4.3449046 , 4.93456784,
       1.32486011, 1.83442209, 1.53671545, 1.79966519, 2.05712637,
       2.3357478 , 1.64996829, 1.65618056, 2.11765717, 4.23013265,
       1.46580203, 1.5159236 , 4.26263726, 4.16297558, 1.8968156 ,
       4.95924072, 2.48265387, 3.67770758, 4.29879246, 1.983562  ,
       1.95071568, 1.63223869, 2.28947596, 1.40411787, 3.41363744,
       1.1301729 , 2.66531179, 1.70714333, 1.97516871, 3.72836914,
       3.6303678 , 1.75373254, 2.12929361, 2.32721171, 1.79835

## Pronósitco usando SGDRegressor

In [11]:
#
# La forma matemática del SGD es un modelo de regresión lineal,
# pero su forma de especificación permite variar la función de
# error, el método de entrenamiento, etc.
#
# En este caso se usa Early Stopping para buscar el mejor ajuste
# a los datos
#
from sklearn.linear_model import SGDRegressor
from sklearn.model_selection import GridSearchCV

parameters = [
    {
        "eta0": [
            0.0005,
            0.0006,
            0.0007,
            0.0008,
            0.0009,
            0.0010,
            0.0011,
            0.0012,
            0.0013,
            0.0014,
        ],
        "power_t": [
            0.14,
            0.15,
            0.16,
            0.17,
            0.18,
            0.19,
            0.20,
        ],
        "random_state": [
            1234568,
            2134568,
            2314568,
            2341568,
            2345168,
            1234668,
            2134668,
            2314668,
            2341668,
            2346168,
            9234568,
            9134568,
            9314568,
            9341568,
            9345168,
            9234668,
            9134668,
            9314668,
            9341668,
            9346168,
            1234568,
            2134568,
            2314568,
            2341568,
            2345168,
            1234668,
            2134668,
            2314668,
            2341668,
            2346167,
            9234567,
            9134567,
            9314567,
            9341567,
            9345167,
            9234667,
            9134667,
            9314667,
            9341667,
            9346167,            
        ],
    },
]

sgdRegressor = SGDRegressor(
    loss="squared_loss",
    penalty="none",
    max_iter=10000000,
    learning_rate="invscaling",
    random_state=1234567,
    early_stopping=True,
    validation_fraction=0.05,
    average=False,
)


sgdreg = GridSearchCV(
    sgdRegressor,
    parameters,
    cv=10,
)

sgdreg.fit(X_train, y_train_true)

y_train_pred = sgdreg.predict(X_train)

mean_squared_error(y_train_true, y_train_pred) / 1000000

7.905252028412068e-07

In [12]:
y_test_pred = sgdreg.predict(X_test)
mean_squared_error(y_test_true, y_test_pred) / 1000000

9.565419470617763e-07

In [13]:
sgdreg.best_estimator_

SGDRegressor(alpha=0.0001, average=False, early_stopping=True, epsilon=0.1,
             eta0=0.0011, fit_intercept=True, l1_ratio=0.15,
             learning_rate='invscaling', loss='squared_loss', max_iter=10000000,
             n_iter_no_change=5, penalty='none', power_t=0.14,
             random_state=9341667, shuffle=True, tol=0.001,
             validation_fraction=0.05, verbose=0, warm_start=False)